In [ ]:
import jsonlines
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
import numpy as np
import nltk
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import regex as re
from collections import Counter

In [2]:
data_list = []

In [3]:
with jsonlines.open("data/train.jsonl", mode='r') as reader:
    for obj in reader:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        data_list.append(obj)

In [4]:
data_list[:5]

[{'text': 'Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.\n\nEllington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestrating his compositions with a keen ear for vocal qualities. This approach is vividly demonstrated in "Mood Indigo," where his orchestration skills shine through non-traditional chord arrangements, transforming the piece into an auditory tapestry of mood and color. It is not merely the notes t

In [5]:
len(data_list)

23707

In [6]:
df = pd.DataFrame(data_list)

In [7]:
print(df)

                                                    text  label
0      Duke Ellington, a titan of jazz, revolutionize...      1
1      I reflected on the shifting dynamics of media ...      1
2      In F. Scott Fitzgerald's "The Great Gatsby," t...      1
3      I still chuckle when I think about that time I...      1
4      Yoga, originating in ancient India, encompasse...      1
...                                                  ...    ...
23702  Antioch stands as a pivotal point in the narra...      1
23703  Margaret Cavendish’s The Blazing World stands ...      1
23704  In August of 1793, in a last-ditch effort to w...      0
23705  In interacting with Luciente and others, Conni...      0
23706  Meursault and Clamence, the protagonists from ...      1

[23707 rows x 2 columns]


In [56]:
type(df.text[0])

str

In [12]:
tokens_all = [word_tokenize(sentence) for sentence in df.text]

In [13]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in tokens_all:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [14]:
print(tokens_all[15000])

['She', 'was', 'small', 'almost', 'childish', 'in', 'stature', 'with', 'a', 'thin', 'erect', 'little', 'figure', 'and', 'a', 'pale', 'oval', 'face', 'framed', 'in', 'short', 'curly', 'hair', 'and', 'at', 'first', 'sight', 'people', 'always', 'called', 'her', 'plain', 'an', 'insignificant', 'puny', 'little', 'thing', 'that', 'was', 'what', 'they', 'said', 'until', 'they', 'saw', 'her', 'eyes', 'and', 'they', 'were', 'the', 'most', 'wonderful', 'and', 'spirituelle', 'eyes', 'in', 'the', 'world', 'And', 'after', 'that', 'they', 'were', 'not', 'so', 'sure', 'of', 'the', 'plainness', 'For', 'comparisons', 'are', 'odious', 'and', 'there', 'is', 'no', 'hard', 'and', 'fast', 'rule', 'with', 'respect', 'to', 'feminine', 'beauty', 'at', 'least', 'tastes', 'differ', 'and', 'here', 'and', 'there', 'a', 'Philistine', 'might', 'be', 'found', 'who', 'would', 'be', 'ready', 'to', 'swear', 'that', 'dark', 'spirituelle', 'eyes', 'brimful', 'of', 'intelligence', 'and', 'animation', 'with', 'a', 'mirthful

In [15]:
word_list = [[word.lower() for word in sentence] for sentence in tokens_all]

In [16]:
len(word_list)

23707

In [ ]:
print(word_list[:2])

['meursault', 'and', 'clamence', 'the', 'protagonists', 'from', 'albert', 'camus', "'s", 'the', 'stranger', 'and', 'patrick', 'kertes', "'s", 'the', 'tenant', 'respectively', 'embody', 'starkly', 'differing', 'attitudes', 'towards', 'isolation', 'and', 'relationships', 'meursault', 'the', 'unassuming', 'and', 'detached', 'protagonist', 'of', 'the', 'stranger', 'is', 'largely', 'unaware', 'of', 'his', 'isolation', 'from', 'society', 'this', 'is', 'evident', 'when', 'he', 'states', 'the', 'sun', 'was', 'the', 'same', 'as', 'is', 'the', 'air', 'and', 'the', 'light', 'and', 'it', 'made', 'me', 'so', 'happy', 'as', 'for', 'the', 'other', 'things', 'i', 'was', 'careful', 'to', 'avoid', 'them', 'camus', '1', 'meursault', "'s", 'indifference', 'stems', 'from', 'a', 'profound', 'existential', 'isolation', 'which', 'continues', 'throughout', 'his', 'life', 'he', 'lives', 'in', 'a', 'state', 'of', 'absurdity', 'disassociated', 'from', 'societal', 'norms', 'and', 'expectations', 'displaying', 'a',

In [18]:
words = []

In [19]:
for sentence in word_list:
    for word in sentence:
        words.append(word)

In [20]:
len(words)

14949644

In [21]:
vocab = sorted(set(words))

In [22]:
len(vocab)

122750

In [23]:
freq = Counter(words)

In [24]:
print(freq)

Counter({'the': 934177, 'of': 509946, 'and': 418215, 'a': 414654, 'to': 369248, 'in': 250901, 'that': 181566, 'his': 162815, 'with': 148387, 'i': 139145, 'was': 139001, 'he': 137382, 'as': 133186, 'her': 129170, 'it': 121503, 'for': 116886, "'s": 96529, 'had': 85666, 'is': 84856, 'on': 79228, 'not': 78460, 'you': 78182, 'she': 76484, 'but': 72102, 'by': 67486, 'at': 65155, 'this': 61339, 'their': 57580, 'be': 57062, 'from': 56160, 'an': 50671, 'have': 50022, 'my': 50010, 'him': 45120, 'they': 42491, 'we': 39539, 's': 39372, 'said': 38386, 'are': 37692, 'were': 37605, 'one': 36764, 'which': 35387, 'who': 34421, 'all': 34122, 'would': 32611, 'me': 31795, 'or': 31132, 'its': 30807, 'so': 30668, 'there': 30435, 'into': 29729, 'been': 29724, 'if': 29015, 'more': 28879, 'like': 27812, 'no': 27646, 'what': 27620, 'could': 26917, 'has': 26226, 'when': 26027, 'our': 25775, 'do': 24065, 'will': 23956, 'them': 23763, 'about': 23702, 'upon': 23199, 'out': 22132, 'can': 21955, 'your': 21047, 'yet':

In [28]:
stop_words = set(stopwords.words('english'))

In [29]:
len(stop_words)

198

In [31]:
filtered_tokens = [word for word in words if word not in stop_words]

In [44]:
# This keeps the structure of a list with all datapoints in the train set, where each datapoint
# is in the form of a list. So it's a list with 23707 lists/data points with tokens.
# Each token has been converted to small characters with word.lower(). 
filtered_token_list = [[token for token in sentence if token not in stop_words] for sentence in word_list]
# Stop-words have now been removed as well. 

In [49]:
print(word_list[0])

['duke', 'ellington', 'a', 'titan', 'of', 'jazz', 'revolutionized', 'the', 'genre', 'through', 'his', 'innovative', 'compositions', 'showcasing', 'a', 'remarkable', 'ability', 'to', 'integrate', 'voice', 'and', 'instrumental', 'music', 'among', 'the', 'notable', 'figures', 'who', 'contributed', 'to', 'this', 'artistic', 'symphony', 'was', 'ivie', 'anderson', 'whose', 'scat', 'singing', 'mirrored', 'the', 'improvisational', 'prowess', 'of', 'musicians', 'like', 'nanton', 'in', 'ring', 'dem', 'bells', 'ellington', 'ingeniously', 'interweaves', 'scat', 'singing', 'as', 'a', 'dialogue', 'with', 'saxophones', 'creating', 'a', 'dynamic', 'call', 'and', 'response', 'that', 'underscores', 'his', 'vision', 'of', 'music', 'as', 'a', 'fluid', 'conversation', 'between', 'voices', 'and', 'instruments', 'ellington', 'consistently', 'emphasized', 'the', 'voice', 'as', 'an', 'instrument', 'of', 'equal', 'importance', 'to', 'traditional', 'brass', 'and', 'woodwinds', 'orchestrating', 'his', 'compositio

In [50]:
print(filtered_token_list[0])

['duke', 'ellington', 'titan', 'jazz', 'revolutionized', 'genre', 'innovative', 'compositions', 'showcasing', 'remarkable', 'ability', 'integrate', 'voice', 'instrumental', 'music', 'among', 'notable', 'figures', 'contributed', 'artistic', 'symphony', 'ivie', 'anderson', 'whose', 'scat', 'singing', 'mirrored', 'improvisational', 'prowess', 'musicians', 'like', 'nanton', 'ring', 'dem', 'bells', 'ellington', 'ingeniously', 'interweaves', 'scat', 'singing', 'dialogue', 'saxophones', 'creating', 'dynamic', 'call', 'response', 'underscores', 'vision', 'music', 'fluid', 'conversation', 'voices', 'instruments', 'ellington', 'consistently', 'emphasized', 'voice', 'instrument', 'equal', 'importance', 'traditional', 'brass', 'woodwinds', 'orchestrating', 'compositions', 'keen', 'ear', 'vocal', 'qualities', 'approach', 'vividly', 'demonstrated', 'mood', 'indigo', 'orchestration', 'skills', 'shine', 'non-traditional', 'chord', 'arrangements', 'transforming', 'piece', 'auditory', 'tapestry', 'mood'

In [57]:
example = filtered_token_list[0]

In [62]:
string = " ".join(example)

In [63]:
print(string)

duke ellington titan jazz revolutionized genre innovative compositions showcasing remarkable ability integrate voice instrumental music among notable figures contributed artistic symphony ivie anderson whose scat singing mirrored improvisational prowess musicians like nanton ring dem bells ellington ingeniously interweaves scat singing dialogue saxophones creating dynamic call response underscores vision music fluid conversation voices instruments ellington consistently emphasized voice instrument equal importance traditional brass woodwinds orchestrating compositions keen ear vocal qualities approach vividly demonstrated mood indigo orchestration skills shine non-traditional chord arrangements transforming piece auditory tapestry mood color merely notes define ellington 's genius rather way orchestrates elements drawing unique talents band members like nanton hodges williams create symphony voice resonates authenticity purpose composition dusk exemplifies ellington 's adeptness captur

In [32]:
len(words)

14949644

In [33]:
len(filtered_tokens)

8018027

In [36]:
print(filtered_tokens[100:110])

['members', 'like', 'nanton', 'hodges', 'williams', 'create', 'symphony', 'voice', 'resonates', 'authenticity']


In [42]:
filtered_vocab = sorted(set(filtered_tokens))

In [43]:
len(filtered_vocab)

122595

In [51]:
tags = pos_tag(filtered_tokens, tagset='universal')

In [52]:
print(tags[:10])

[('duke', 'ADJ'), ('ellington', 'NOUN'), ('titan', 'NOUN'), ('jazz', 'NOUN'), ('revolutionized', 'VERB'), ('genre', 'ADJ'), ('innovative', 'ADJ'), ('compositions', 'NOUN'), ('showcasing', 'VERB'), ('remarkable', 'ADJ')]


In [54]:
type(tags)

list

In [65]:
tag_list = [pos_tag(sentence, tagset='universal') for sentence in filtered_token_list]

In [66]:
len(tag_list)

23707

In [69]:
print(tag_list[0])

[('duke', 'ADJ'), ('ellington', 'NOUN'), ('titan', 'NOUN'), ('jazz', 'NOUN'), ('revolutionized', 'VERB'), ('genre', 'ADJ'), ('innovative', 'ADJ'), ('compositions', 'NOUN'), ('showcasing', 'VERB'), ('remarkable', 'ADJ'), ('ability', 'NOUN'), ('integrate', 'ADJ'), ('voice', 'NOUN'), ('instrumental', 'ADJ'), ('music', 'NOUN'), ('among', 'ADP'), ('notable', 'ADJ'), ('figures', 'NOUN'), ('contributed', 'VERB'), ('artistic', 'ADJ'), ('symphony', 'NOUN'), ('ivie', 'NOUN'), ('anderson', 'NOUN'), ('whose', 'PRON'), ('scat', 'NOUN'), ('singing', 'NOUN'), ('mirrored', 'VERB'), ('improvisational', 'ADJ'), ('prowess', 'NOUN'), ('musicians', 'NOUN'), ('like', 'ADP'), ('nanton', 'NOUN'), ('ring', 'VERB'), ('dem', 'NOUN'), ('bells', 'NOUN'), ('ellington', 'VERB'), ('ingeniously', 'ADV'), ('interweaves', 'NOUN'), ('scat', 'VERB'), ('singing', 'VERB'), ('dialogue', 'NOUN'), ('saxophones', 'NOUN'), ('creating', 'VERB'), ('dynamic', 'ADJ'), ('call', 'ADJ'), ('response', 'NOUN'), ('underscores', 'VERB'), (

In [76]:
print(tag_list[0][0][-1])

ADJ


In [77]:
sentence_tags = [[token[-1] for token in sentence] for sentence in tag_list]

In [82]:
print(sentence_tags[:2])

[['ADJ', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'PRON', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'ADP', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADV', 'NOUN', 'VERB', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'ADV', 'VERB', 'ADJ', 'NOUN', 'PRT', 'NOUN', 'ADV', 'NOUN', 'VERB', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'PRT', 'ADJ', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'ADJ', 'VERB', 'A

In [83]:
tags_freq = [[Counter(sentence)] for sentence in sentence_tags]

In [89]:
print(tags_freq[0][0])

Counter({'NOUN': 112, 'ADJ': 50, 'VERB': 47, 'ADV': 7, 'ADP': 6, 'PRT': 3, 'PRON': 1})


In [91]:
example_tags = tags_freq[0][0]

In [92]:
print(example_tags)

Counter({'NOUN': 112, 'ADJ': 50, 'VERB': 47, 'ADV': 7, 'ADP': 6, 'PRT': 3, 'PRON': 1})


In [103]:
print(example_tags.values())

dict_values([50, 112, 47, 6, 1, 7, 3])


In [14]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=5, norm='l2', encoding="utf-8", ngram_range=(1, 2), stop_words='english')

features = tfidf.fit_transform(df.text).toarray()
labels = df.label

In [ ]:
features.shape # Now each of the 23707 sequences of text is represented by 195904 features, 
               # representing the tf-idf score for different unigrams and bigrams.

(23707, 195904)

In [28]:
model = LogisticRegression(random_state=42)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [31]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [34]:
print(metrics.classification_report(y_test, y_pred, labels=df['label']))

              precision    recall  f1-score   support

           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           0       0.97      0.96      0.97      1856
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97    